# Frames and Packing

A **frame** is a special kind of GTree whose layout grows dynamically
as you pack grobs into it. Frames are the grid_py mechanism for
building complex composite objects (such as legends) without having
to precompute the layout.

This notebook covers:

* Creating frames with `frame_grob`.
* Packing grobs into a frame with `pack_grob`.
* Placing grobs into existing cells with `place_grob`.
* A practical example: building a simple legend layout.

The narrative follows the original R vignette (`frame.Rnw`).

In [ ]:
import matplotlib
matplotlib.use("Agg")

import grid_py as gp

## Creating a Frame

`frame_grob()` creates an empty frame -- a GTree with class `"frame"`.
At this point the frame has no layout; one will be created
automatically when grobs are packed in.

In [ ]:
frame = gp.frame_grob(name="my_frame")
print(frame)
print("grid class:", frame._grid_class)

## Packing Grobs

`pack_grob` adds a grob to a frame by specifying which *side*
(`"top"`, `"bottom"`, `"left"`, `"right"`) the new grob should
occupy. Each call extends the frame's internal layout.

In [ ]:
# Pack a title at the top
title = gp.text_grob("My Title", gp=gp.Gpar(fontsize=16), name="title")
frame = gp.pack_grob(frame, title, side="top")

# Pack a coloured rectangle on the left
swatch = gp.rect_grob(
    width=gp.Unit(1, "cm"),
    height=gp.Unit(1, "cm"),
    gp=gp.Gpar(fill="steelblue"),
    name="swatch",
)
frame = gp.pack_grob(frame, swatch, side="left")

# Pack a label on the right
label = gp.text_grob("Category A", gp=gp.Gpar(fontsize=12), name="label")
frame = gp.pack_grob(frame, label, side="right")

print("Children after packing:", gp.child_names(frame))

In [ ]:
# Draw the packed frame
gp.grid_newpage()
gp.grid_draw(frame)

## Placing Grobs

`place_grob` puts a grob into an *existing* cell of the frame's layout
(it does not add new rows or columns). This is useful when you want to
overlay or fill a specific cell.

In [ ]:
# Create a frame with an explicit layout so we know the cell positions
layout = gp.GridLayout(nrow=2, ncol=2)
frame2 = gp.frame_grob(layout=layout, name="placed_frame")

# Place grobs into specific cells
frame2 = gp.place_grob(
    frame2,
    gp.rect_grob(gp=gp.Gpar(fill="salmon"), name="r1"),
    row=1, col=1,
)
frame2 = gp.place_grob(
    frame2,
    gp.rect_grob(gp=gp.Gpar(fill="lightblue"), name="r2"),
    row=1, col=2,
)
frame2 = gp.place_grob(
    frame2,
    gp.text_grob("A", name="t1"),
    row=2, col=1,
)
frame2 = gp.place_grob(
    frame2,
    gp.text_grob("B", name="t2"),
    row=2, col=2,
)

gp.grid_newpage()
gp.grid_draw(frame2)

## Building a Legend Layout

Frames shine when building complex composite objects like legends.
The following example constructs a simple two-item legend by
repeatedly packing colour swatches and labels.

In [ ]:
colours = ["firebrick", "forestgreen"]
labels  = ["Group 1", "Group 2"]

legend = gp.frame_grob(name="legend")

# Add a legend title
legend = gp.pack_grob(
    legend,
    gp.text_grob("Legend", gp=gp.Gpar(fontsize=14, fontface="bold"), name="legend_title"),
    side="top",
)

# For each item, pack swatch on the left and label on the right
for i, (col, lab) in enumerate(zip(colours, labels)):
    row_frame = gp.frame_grob(name=f"row_{i}")
    row_frame = gp.pack_grob(
        row_frame,
        gp.rect_grob(
            width=gp.Unit(0.5, "cm"),
            height=gp.Unit(0.5, "cm"),
            gp=gp.Gpar(fill=col),
            name=f"swatch_{i}",
        ),
        side="left",
    )
    row_frame = gp.pack_grob(
        row_frame,
        gp.text_grob(lab, gp=gp.Gpar(fontsize=11), name=f"label_{i}"),
        side="right",
    )
    legend = gp.pack_grob(legend, row_frame, side="top")

gp.grid_newpage()
gp.grid_draw(legend)

## Summary

* `frame_grob` creates an empty frame (a GTree with a dynamic layout).
* `pack_grob` adds grobs to a frame, growing the layout row by row or
  column by column via the `side` argument.
* `place_grob` places grobs into existing layout cells.
* Frames are the idiomatic way to build legends and other composite
  graphical objects in grid_py.